In [1]:
import json
import os
import glob
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve
from catboost import CatBoostClassifier

# 📌 외부 파일에서 딥러닝 모델 클래스 임포트
from models import ChurnLSTM, ChurnCNN1D, ChurnTransformer

In [ ]:
# ==========================================
# 🚀 1. 디렉토리 구조 설정 및 테스트셋 준비
# ==========================================
DATA_DIR = "./data/data/processed"
ML_MODEL_DIR = "./model/machine_learning"
DL_MODEL_DIR = "./model/deep_learning"
EVAL_DIR = "./test"
TEST_CSV_PATH = "./data/data/processed/test_20181101_v1.csv"  # 테스트용 정형 데이터 파일
TEST_NPY_PATH = "./data/data/processed/test.npy"

os.makedirs(EVAL_DIR, exist_ok=True)

print("=" * 50)
print("🚀 1. 테스트셋 데이터 로드 및 준비 (개별 평가 전용)")
print("=" * 50)

# 1. 테스트 정형 데이터 및 시퀀스 텐서 직접 로드
df_test = pd.read_csv(TEST_CSV_PATH)

if os.path.exists(TEST_NPY_PATH):
    X_seq_test = np.load(TEST_NPY_PATH)
else:
    raise FileNotFoundError(f"'{TEST_NPY_PATH}' 파일을 찾을 수 없습니다. 시퀀스 변환 코드를 먼저 실행해 주세요.")

# 타겟 컬럼 지정
target_col = "churn" if "churn" in df_test.columns else ("is_churn" if "is_churn" in df_test.columns else df_test.columns[-1])
y_test = df_test[target_col].values
y_true = np.array(y_test).ravel()

# 🛠️ 불필요한 컬럼 제거 및 피처 추출
drop_cols = [target_col, "user_id", "userId", "page_sequence", "session_item_sequence", "padded_sequence"]
feature_cols = [col for col in df_test.columns if col not in drop_cols]
X_tab_test = df_test[feature_cols].copy()

# 범주형 컬럼 인코딩
for col in ["gender", "level", "last_page", "level_last"]:
    if col in X_tab_test.columns:
        X_tab_test[col] = X_tab_test[col].astype("category").cat.codes

# 객체 타입 컬럼 숫자형 변환 및 결측치 처리
for col in X_tab_test.columns:
    if X_tab_test[col].dtype == "object":
        X_tab_test[col] = pd.to_numeric(X_tab_test[col], errors="coerce").fillna(0.0)

print(f"✅ X_tab_test 정비 완료! 최종 shape: {X_tab_test.shape} (피처 수: {X_tab_test.shape[1]}개)")
print(f"✅ X_seq_test 텐서 shape: {X_seq_test.shape}")

# 딥러닝(DL) 테스트 데이터로더 구성
test_dataset = TensorDataset(torch.tensor(X_seq_test, dtype=torch.long), torch.tensor(y_test, dtype=torch.float32))
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

🚀 1. 테스트셋 데이터 로드 및 준비 (개별 평가 전용)


FileNotFoundError: [Errno 2] No such file or directory: './data/data/processedtest_20181101_v1.csv'

In [ ]:
# ==========================================
# 🚀 2. 모델 디렉토리 스캔 및 예측값 수집 (안전한 분기 처리)
# ==========================================
print("\n" + "=" * 50)
print("🚀 2. 저장된 모델 스캔 및 예측 수행")
print("=" * 50)

# 2.1 머신러닝 모델 예측값 수집 (.pkl 및 .cbm 파일 동시 지원)
ml_predictions = {}
ml_model_files = glob.glob(os.path.join(ML_MODEL_DIR, "*.pkl")) + glob.glob(os.path.join(ML_MODEL_DIR, "*.cbm"))

if not ml_model_files:
    print(f"⚠️ 경고: {ML_MODEL_DIR} 경로에 저장된 머신러닝 모델(.pkl, .cbm)이 없습니다.")

for model_path in ml_model_files:
    model_name = os.path.basename(model_path)
    
    if model_name.endswith(".pkl"):
        model = joblib.load(model_path)
    elif model_name.endswith(".cbm"):
        model = CatBoostClassifier()
        model.load_model(model_path)
    else:
        raise ValueError(f"지원하지 않는 머신러닝 모델 형식입니다: {model_name}")
        
    # 💡 [핵심 해결] 모델 종류에 따른 안전한 예측 수행
    try:
        # CatBoost 모델이거나 n_features_in_ 속성이 있는 경우
        if isinstance(model, CatBoostClassifier) or hasattr(model, "n_features_in_"):
            n_feat = getattr(model, "n_features_in_", X_tab_test.shape[1])
            # 학습 피처 개수에 맞게 슬라이싱하여 전달
            preds = model.predict_proba(X_tab_test.iloc[:, :n_feat])[:, 1]
        else:
            preds = model.predict_proba(X_tab_test)[:, 1]
    except Exception as e:
        # LightGBM 등에서 피처 검증 에러가 날 경우 validate_features=False 적용
        try:
            preds = model.predict_proba(X_tab_test, validate_features=False)[:, 1]
        except Exception as e2:
            # 마지막 안전장치: 강제로 numpy array로 변환 후 예측
            preds = model.predict_proba(X_tab_test.values)[:, 1]
            
    ml_predictions[model_name] = preds
    print(f"📥 [ML 로드 완료] {model_name} (Test AUC: {roc_auc_score(y_true, preds):.4f})")

# 2.2 딥러닝 모델 예측값 수집
dl_predictions = {}
dl_model_files = glob.glob(os.path.join(DL_MODEL_DIR, "*.pth"))
if not dl_model_files:
    print(f"⚠️ 경고: {DL_MODEL_DIR} 경로에 저장된 딥러닝 모델 가중치(.pth)가 없습니다.")

for model_path in dl_model_files:
    model_name = os.path.basename(model_path)
    
    if "lstm" in model_name.lower():
        dl_model = ChurnLSTM(vocab_size=50, embedding_dim=32, hidden_dim=64).to(device)
    elif "cnn" in model_name.lower():
        dl_model = ChurnCNN1D(vocab_size=50, embedding_dim=32, num_filters=64).to(device)
    elif "transformer" in model_name.lower():
        dl_model = ChurnTransformer(vocab_size=50, embedding_dim=32, num_heads=4, hidden_dim=64, num_layers=2, dropout=0.1).to(device)
    else:
        raise ValueError(f"지원하지 않는 딥러닝 모델 파일명입니다: {model_name}")
        
    dl_model.load_state_dict(torch.load(model_path, map_location=device))
    dl_model.eval()
    
    dl_preds_list = []
    with torch.no_grad():
        for seqs, _ in test_loader:
            outputs = torch.sigmoid(dl_model(seqs.to(device)))
            dl_preds_list.extend(outputs.cpu().numpy())
    
    preds = np.array(dl_preds_list).ravel()
    dl_predictions[model_name] = preds
    print(f"📥 [DL 로드 완료] {model_name} (Test AUC: {roc_auc_score(y_true, preds):.4f})")


🚀 2. 저장된 모델 스캔 및 예측 수행
📥 [ML 로드 완료] ab_model.pkl (Test AUC: 0.7145)
📥 [ML 로드 완료] lgb_model.pkl (Test AUC: 0.7232)
📥 [ML 로드 완료] lr_model.pkl (Test AUC: 0.7095)
📥 [ML 로드 완료] rf_model.pkl (Test AUC: 0.7084)
📥 [ML 로드 완료] catboost_model.cbm (Test AUC: 0.7210)
📥 [DL 로드 완료] cnn1d_model_weights.pth (Test AUC: 0.8224)
📥 [DL 로드 완료] lstm_model_weights.pth (Test AUC: 0.8519)
📥 [DL 로드 완료] transformer_model_weights.pth (Test AUC: 0.8338)


In [ ]:
# ==========================================
# 🚀 3. 각 모델별 개별 성능 평가 및 리포트 출력
# ==========================================
print("\n" + "=" * 50)
print("🚀 3. 개별 모델 성능 평가 리포트 출력 (F2-Score 기준)")
print("=" * 50)

beta = 2.0  # 재현율에 가중치를 부여하는 F2-Score 설정
evaluation_results = {}

# 3.1 머신러닝(ML) 모델 개별 평가 루프
print("\n📌 [ 머신러닝(ML) 모델별 상세 리포트 ]")
for model_name, preds in ml_predictions.items():
    auc_score = roc_auc_score(y_true, preds)
    
    precisions, recalls, thresholds = precision_recall_curve(y_true, preds)
    p, r, t = precisions[:-1], recalls[:-1], thresholds
    f_beta_scores = (1 + beta**2) * (p * r) / ((beta**2 * p) + r + 1e-10)
    
    best_idx = np.argmax(f_beta_scores)
    best_threshold = t[best_idx]
    optimal_preds = (preds >= best_threshold).astype(int)
    
    print(f"\n--------------------------------------------------")
    print(f"🔹 ML 모델 이름: {model_name}")
    print(f"🎯 Test AUC: {auc_score:.4f} | 재현율 우선 최적 임계값(F2): {best_threshold:.4f}")
    print("--------------------------------------------------")
    print(classification_report(y_true, optimal_preds))
    
    # 개별 예측값 및 요약 정보 저장
    np.save(os.path.join(EVAL_DIR, f"pred_{model_name}.npy"), preds)
    evaluation_results[model_name] = {
        "model_type": "machine_learning",
        "auc_score": float(auc_score),
        "best_threshold": float(best_threshold)
    }

# 3.2 딥러닝(DL) 모델 개별 평가 루프
print("\n📌 [ 딥러닝(DL) 모델별 상세 리포트 ]")
for model_name, preds in dl_predictions.items():
    auc_score = roc_auc_score(y_true, preds)
    
    precisions, recalls, thresholds = precision_recall_curve(y_true, preds)
    p, r, t = precisions[:-1], recalls[:-1], thresholds
    f_beta_scores = (1 + beta**2) * (p * r) / ((beta**2 * p) + r + 1e-10)
    
    best_idx = np.argmax(f_beta_scores)
    best_threshold = t[best_idx]
    optimal_preds = (preds >= best_threshold).astype(int)
    
    print(f"\n--------------------------------------------------")
    print(f"🔹 DL 모델 이름: {model_name}")
    print(f"🎯 Test AUC: {auc_score:.4f} | 재현율 우선 최적 임계값(F2): {best_threshold:.4f}")
    print("--------------------------------------------------")
    print(classification_report(y_true, optimal_preds))
    
    # 개별 예측값 및 요약 정보 저장
    np.save(os.path.join(EVAL_DIR, f"pred_{model_name}.npy"), preds)
    evaluation_results[model_name] = {
        "model_type": "deep_learning",
        "auc_score": float(auc_score),
        "best_threshold": float(best_threshold)
    }

# 전체 개별 모델 요약 설정 파일 저장
with open(os.path.join(EVAL_DIR, "individual_models_evaluation.json"), "w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, ensure_ascii=False, indent=4)

print(f"\n✨ 모든 개별 모델의 예측값(`.npy`)과 평가 요약 파일(`individual_models_evaluation.json`)이 [{EVAL_DIR}] 디렉토리에 성공적으로 저장되었습니다!")


🚀 3. 개별 모델 성능 평가 리포트 출력 (F2-Score 기준)

📌 [ 머신러닝(ML) 모델별 상세 리포트 ]

--------------------------------------------------
🔹 ML 모델 이름: ab_model.pkl
🎯 Test AUC: 0.7145 | 재현율 우선 최적 임계값(F2): 0.3094
--------------------------------------------------
              precision    recall  f1-score   support

           0       0.97      0.61      0.75      3421
           1       0.10      0.71      0.17       207

    accuracy                           0.61      3628
   macro avg       0.54      0.66      0.46      3628
weighted avg       0.92      0.61      0.72      3628


--------------------------------------------------
🔹 ML 모델 이름: lgb_model.pkl
🎯 Test AUC: 0.7232 | 재현율 우선 최적 임계값(F2): 0.0633
--------------------------------------------------
              precision    recall  f1-score   support

           0       0.97      0.72      0.83      3421
           1       0.12      0.60      0.20       207

    accuracy                           0.72      3628
   macro avg       0.54      0.66     